# Metric Presentation and Visualization
## Necessary packages and functions call

- DDPM-TS: Interpretable Diffusion for Time Series Generation
- Metrics: 
    - discriminative_metrics
    - predictive_metrics
    - visualization

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
#os.environ['TF_USE_LEGACY_KERAS'] = '1'

import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

## Data Loading

Load original dataset and preprocess the loaded data.

In [ ]:
iterations = 5
#ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
#fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')

#UNCONDITIONAL GENERATION
'''
ori_data = np.load('../OUTPUT/etth/samples/etth_norm_truth_24_train.npy') #Need to use normalized samples!
fake_data = np.load('../OUTPUT/etth/ddpm_fake_etth.npy')
'''

#PREDICTION

ori_data = np.load('../OUTPUT/sines/samples/sine_ground_truth_24_test.npy')
fake_data = np.load('../OUTPUT/sines/ddpm_predict-counterfactual_sines_l1_l2.npy')

## Evaluate the generated data

### 1. Discriminative score

To evaluate the classification accuracy between original and synthetic data using post-hoc RNN network. The output is | classification accuracy - 0.5 |.

- metric_iteration: the number of iterations for metric computation.

In [56]:
discriminative_score = []

for i in range(iterations):
    temp_disc, fake_acc, real_acc = discriminative_score_metrics(ori_data[:], fake_data[:ori_data.shape[0]])
    discriminative_score.append(temp_disc)
    print(f'Iter {i}: ', temp_disc, ',', fake_acc, ',', real_acc, '\n')
      
print('sine:')
display_scores(discriminative_score)
print()

training:   0%|          | 0/2000 [00:00<?, ?it/s]

Iter 0:  0.05249999999999999 , 0.595 , 0.51 



training:   0%|          | 0/2000 [00:00<?, ?it/s]

Iter 1:  0.030000000000000027 , 0.665 , 0.395 



training:   0%|          | 0/2000 [00:00<?, ?it/s]

Iter 2:  0.015000000000000013 , 0.575 , 0.455 



training:   0%|          | 0/2000 [00:00<?, ?it/s]

Iter 3:  0.08250000000000002 , 0.61 , 0.555 



training:   0%|          | 0/2000 [00:00<?, ?it/s]

Iter 4:  0.0625 , 0.665 , 0.46 

sine:
Final Score:  0.04850000000000001 ± 0.0330414465658783



## Evaluate the generated data

### 2. Predictive score

To evaluate the prediction performance on train on synthetic, test on real setting. More specifically, we use Post-hoc RNN architecture to predict one-step ahead and report the performance in terms of MAE. 

The model learns to predict the last dimension with one more step.

In [60]:
predictive_score = []
for i in range(iterations):
    temp_pred = predictive_score_metrics(ori_data, fake_data[:ori_data.shape[0]])
    predictive_score.append(temp_pred)
    print(i, ' epoch: ', temp_pred, '\n')
      
print('sine:')
display_scores(predictive_score)
print()

training:   0%|          | 0/5000 [00:00<?, ?it/s]

0  epoch:  0.09913106978830544 



training:   0%|          | 0/5000 [00:00<?, ?it/s]

1  epoch:  0.09877967552739442 



training:   0%|          | 0/5000 [00:00<?, ?it/s]

2  epoch:  0.09904722460894458 



training:   0%|          | 0/5000 [00:00<?, ?it/s]

3  epoch:  0.09855089711395112 



training:   0%|          | 0/5000 [00:00<?, ?it/s]

4  epoch:  0.09881045972888758 

sine:
Final Score:  0.09886386535349663 ± 0.00028646404313917686

